# Ollama GPU Setup — Google Colab T4

Runs `qwen2.5:7b` on a free T4 GPU and exposes it to your local Next.js app via a cloudflared tunnel.

**Before running:** Enable GPU runtime → Runtime → Change runtime type → T4 GPU

**Run cells in order, top to bottom.**

## Cell 1 — Install Ollama

In [ ]:
import subprocess

result = subprocess.run(
    'curl -fsSL https://ollama.com/install.sh | sh',
    shell=True, capture_output=True, text=True
)
print(result.stdout[-800:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-300:])

## Cell 2 — Start Ollama

Colab has no systemd. Start Ollama as a subprocess with `OLLAMA_ORIGINS=*` so the cloudflared tunnel is allowed through.

In [ ]:
import subprocess, time, os, requests

# Kill any existing instance
subprocess.run(['pkill', '-9', '-f', 'ollama'], capture_output=True)
time.sleep(2)

env = os.environ.copy()
env['OLLAMA_ORIGINS'] = '*'
env['OLLAMA_HOST'] = '0.0.0.0:11434'

proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env
)
time.sleep(4)

r = requests.get('http://localhost:11434/api/tags')
print('✓ Ollama running:', r.status_code)

## Cell 3 — Pull model + verify

In [ ]:
import subprocess, requests

result = subprocess.run(['ollama', 'pull', 'qwen2.5:7b'], capture_output=True, text=True)
print(result.stdout[-300:] or result.stderr[-300:])

r = requests.get('http://localhost:11434/api/tags')
models = [m['name'] for m in r.json().get('models', [])]
print('Models available:', models)

# Quick smoke test
r2 = requests.post('http://localhost:11434/api/generate',
    json={'model': 'qwen2.5:7b', 'prompt': 'hi', 'stream': False})
print('Smoke test response:', r2.json().get('response', ''))

## Cell 4 — Install cloudflared

In [ ]:
import subprocess

subprocess.run([
    'wget', '-q',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    '-O', '/usr/local/bin/cloudflared'
], check=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

v = subprocess.run(['cloudflared', '--version'], capture_output=True, text=True)
print('✓ cloudflared:', v.stdout.strip() or v.stderr.strip())

## Cell 5 — Start Host-header proxy on port 11435

Ollama rejects requests whose `Host` header doesn't match `localhost` (DNS-rebinding protection).  
The cloudflared tunnel sends its own domain as `Host`, which causes a 403.  
This proxy sits between cloudflared and Ollama, strips the `Host` header, and forwards cleanly.

In [ ]:
import threading
from http.server import HTTPServer, BaseHTTPRequestHandler
import urllib.request

class OllamaProxy(BaseHTTPRequestHandler):
    def do_request(self):
        length = int(self.headers.get('Content-Length', 0))
        body = self.rfile.read(length) if length > 0 else None
        headers = {k: v for k, v in self.headers.items()
                   if k.lower() not in ('host', 'content-length')}
        if body:
            headers['Content-Length'] = str(len(body))
        try:
            req = urllib.request.Request(
                f'http://localhost:11434{self.path}',
                data=body, headers=headers, method=self.command
            )
            with urllib.request.urlopen(req, timeout=300) as resp:
                self.send_response(resp.status)
                for k, v in resp.headers.items():
                    if k.lower() != 'transfer-encoding':
                        self.send_header(k, v)
                self.end_headers()
                while chunk := resp.read(4096):
                    self.wfile.write(chunk)
        except Exception as e:
            self.send_error(502, str(e))

    do_GET = do_POST = do_PUT = do_DELETE = do_request
    def log_message(self, *_): pass

server = HTTPServer(('0.0.0.0', 11435), OllamaProxy)
threading.Thread(target=server.serve_forever, daemon=True).start()
print('✓ Proxy running on :11435')

## Cell 6 — Start cloudflared tunnel

Tunnels to port **11435** (the proxy), not 11434 directly.  
Copy the printed URL into `.env.local` as `OLLAMA_BASE_URL`, then restart `npm run dev`.

In [ ]:
import subprocess, threading, time, re

subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:11435'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

tunnel_url = None

def read():
    global tunnel_url
    for line in proc.stdout:
        print(line, end='')
        if 'trycloudflare.com' in line and not tunnel_url:
            m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
            if m:
                tunnel_url = m.group(0)
                print(f'\n✓ Tunnel URL: {tunnel_url}')
                print(f'\n→ Set in .env.local:  OLLAMA_BASE_URL={tunnel_url}')

threading.Thread(target=read, daemon=False).start()
time.sleep(10)

## Cell 7 — Health check

Run after updating `.env.local`. All three should return 200.

In [ ]:
import requests, subprocess

# Ollama direct
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=5)
    print('✓ Ollama direct:', r.status_code)
except Exception as e:
    print('✗ Ollama down:', e)

# Proxy
try:
    r = requests.get('http://localhost:11435/api/tags', timeout=5)
    print('✓ Proxy (11435):', r.status_code)
except Exception as e:
    print('✗ Proxy down:', e)

# cloudflared process
result = subprocess.run(['pgrep', '-a', '-f', 'cloudflared'], capture_output=True, text=True)
if result.stdout.strip():
    print('✓ cloudflared running:', result.stdout.strip().split('\n')[0])
else:
    print('✗ cloudflared NOT running — re-run Cell 6')

# Tunnel (replace with your current URL)
if tunnel_url:
    try:
        r = requests.get(f'{tunnel_url}/api/tags', timeout=10)
        print('✓ Tunnel reachable:', r.status_code, '(200 = all good, 403 = re-run Cell 2)')
    except Exception as e:
        print('✗ Tunnel unreachable:', e)
else:
    print('⚠ tunnel_url not set — re-run Cell 6')

## Cell 8 — Keep-alive

Colab disconnects after ~90 min of inactivity. Keep this cell running while using the chatbot.  
Interrupt it (square stop button) when done.

In [ ]:
import time, requests

print('Keep-alive running — interrupt this cell when done')
while True:
    try:
        requests.get('http://localhost:11434/api/tags', timeout=5)
    except Exception:
        print('⚠ Ollama not responding — may need restart')
    time.sleep(30)